# LUMIERE same-week pairs: manifest, mask check, split, registration, export

EXP-2026-005, steps 2 and 4 to 7, on the 36 same-week pairs that have a pre-op tumor mask.

- Reads LUMIERE from `MyDrive/LUMIERE` through the mount and never writes there.
- Writes its records to `MyDrive/LUMIERE_derived/EXP-2026-005/<RUN_TAG>` through the Drive API; every file is confirmed by the server's MD5. These records hold patient names: keep that folder private.
- Prints counts only.
- No GPU needed: a CPU runtime is enough. On the laptop the same code took 155 s for 4 pairs. For 36 pairs on Colab I expect roughly 30 to 60 minutes (not measured). Keep the tab open until the last cell finishes.

In [ ]:
REPO_URL = "https://github.com/AIVIETNAM-AIO-Tuan/MedSegFactory.git"
BRANCH = "lumiere-onboarding"
WORK = "/content/MedSegFactory-lumiere"
LUMIERE_ROOT = "/content/drive/MyDrive/LUMIERE"
WORKERS = 16  # parallel folder listings on the Drive mount
INVENTORY = "/content/lumiere_inventory.json"  # reused if an earlier LUMIERE notebook ran in this runtime
RUN_TAG = "r1"  # use a new tag for every attempt; a run folder is never reused
DRIVE_RUN = ("LUMIERE_derived", "EXP-2026-005", RUN_TAG)  # created under MyDrive through the Drive API
LOCAL_RUN = f"/content/lumiere_exp005_{RUN_TAG}"  # the originals; uploaded to Drive as the run goes

In [ ]:
import json
import os
import subprocess
import sys
import time
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")
assert Path(LUMIERE_ROOT).is_dir(), f"{LUMIERE_ROOT} not found: check the folder name"
assert not Path(LOCAL_RUN).exists(), f"{LOCAL_RUN} exists: choose a new RUN_TAG"
if not Path(WORK).exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, WORK], check=True)
else:
    current = subprocess.check_output(["git", "-C", WORK, "branch", "--show-current"], text=True).strip()
    assert current == BRANCH, "This checkout belongs to another task; choose a different WORK"
    subprocess.run(["git", "-C", WORK, "pull", "--ff-only"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "SimpleITK==2.5.6", "nibabel>=5.3,<6"], check=True)
os.chdir(WORK)
sys.path.insert(0, WORK)
print(subprocess.check_output(["git", "log", "-1", "--oneline"], text=True).strip())

from google.colab import auth
from googleapiclient.discovery import build

from rhuh_synthesis.drive_api import DriveRun

auth.authenticate_user()  # Colab asks once for permission to use the Drive API with this account
run_drive = DriveRun.open(build("drive", "v3", cache_discovery=False), (), DRIVE_RUN, LOCAL_RUN, create=True)
access = run_drive.preflight()  # a real write, update, read and removal on Drive before the long run
print("free GiB on Drive:", None if access["free_gib"] is None else round(access["free_gib"], 1))
assert access["free_gib"] is None or access["free_gib"] > 2, "About 0.5 GB will be written: free some Drive space first"

In [ ]:
from rhuh_synthesis import lumiere as L
from rhuh_synthesis import lumiere_pairs as P

if Path(INVENTORY).exists() and json.loads(Path(INVENTORY).read_text())["root"] == LUMIERE_ROOT:
    inventory = json.loads(Path(INVENTORY).read_text())
    print("inventory reused from this runtime")
else:
    inventory = L.scan(LUMIERE_ROOT, workers=WORKERS, progress=print)  # about 3 minutes
    Path(INVENTORY).write_text(json.dumps(inventory))
tables, table_check = L.resolve_public_tables(LUMIERE_ROOT, inventory["tables"], "/content")
ratings, completeness = L.read_ratings(tables["ratings"]), L.read_completeness(tables["completeness"])
pairs = P.eligible_pairs(inventory, ratings, completeness)
pinned = [pair["patient"] for pair in P.choose_sample(pairs, seed=2026)]  # the 4 pairs on the laptop: training role
print(table_check)
print("eligible same-week pairs:", len(pairs), "| pinned to training:", len(pinned))
assert len(pairs) == 36, "The geometry run found 36 pairs: stop here and send this output"

In [ ]:
from rhuh_synthesis.lumiere_export import run_package_a

config = json.loads(Path("configs/experiments/EXP-2026-001.json").read_text(encoding="utf-8"))  # frozen, read only
settings = json.loads(Path("configs/experiments/EXP-2026-005.json").read_text(encoding="utf-8"))
started = time.time()


def say(message):
    print(f"[{time.time() - started:6.0f} s] {message}", flush=True)


def sync():
    report = run_drive.sync()
    if report["failed"]:
        say(f"Drive: {len(report['failed'])} files not uploaded yet; they are retried later")


summary = run_package_a(LUMIERE_ROOT, pairs, config, settings, Path(LOCAL_RUN) / "outputs", pinned=pinned,
                        repo_root=WORK, progress=say, sync=sync)
say("done" if summary["stopped_at"] is None else f"stopped at {summary['stopped_at']}")

In [ ]:
assert run_drive.confirm(), f"Not every file is confirmed on Drive. The originals are in {LOCAL_RUN}: keep this session open and report"
print("Every file confirmed by the Drive server: MyDrive/" + "/".join(DRIVE_RUN))
print("SUMMARY")
print(json.dumps(summary, indent=1))

## What to send back

Copy everything printed after `SUMMARY`. It holds counts only. Send it also when `stopped_at` is not `null`: the run then stopped on purpose at one of the card's controls.

If a cell stops with an error, send the error message instead.

Registration overlays of the training-role pairs are on Drive in `outputs/qc_overlays/train`. Look at them only for problems that repeat across pairs; a pair's acceptance is never changed by looking at it. No overlays are written for the development pairs.